### Transformer Improvements

#### Local/sparse attention

Reduces computational complexity by limiting attention to nearby tokens, significantly improving efficiency for long sequences at cost of some context awareness.

  - Local (sliding window): Each token attends only to a fixed number of neighboring tokens.

    ![image.png](./951c27af_image.png)

  - Sparse: Attention is computed only for a subset of token pairs based on a predefined pattern.

    ![image-2.png](./951c27af_image-2.png)  

  - Global tokens: Introduce special tokens that can attend to all other tokens, providing a way to capture global context without full attention.
  - Random or block sparse patterns: Use random or structured sparsity patterns to reduce the number of attention computations while still capturing important relationships.

#### Multi-Query Attention (MQA)

MQA keeps *one* set of keys/values for **all** attention heads, while still letting each head have its own query matrix.  
    This saves memory, speeds up attention, and keeps the same expressive power as multi-head attention.

  | Step | What happens | Why it’s useful |
  |------|--------------|-----------------|
  | 1️⃣  **Split into heads** | The model splits its hidden dimension into, say, 8 heads. | Each head can focus on a different pattern or feature. |
  | 2️⃣  **Compute Q, K, V for every head** | Each head gets its own **query (Q)**, **key (K)**, and **value (V)** matrices. | Gives each head a fresh “view” of the data. |
  | 3️⃣  **Do attention per head** | Head i attends to head i’s K/V. | Captures many types of interactions in parallel. |

  *Result*: For a sequence of length L, we do **L × L** pairwise dot‑products **per head**.  That’s the classic quadratic cost.

  | Difference | How MQA does it | What it buys us |
  |------------|-----------------|-----------------|
  | **Key/Value** | **Shared** across all heads | Fewer K/V tensors → less memory and less computation |
  | **Query** | Still **one per head** | Each head can still look in a different direction |
  | **Attention matrix** | Same for all heads (because K/V are shared) | We compute it once and reuse it, so we avoid the per‑head cost |

  ![image-3.png](./951c27af_image-3.png)

  - **Q** matrices are separate (different heads).  
  - **K/V** are the same for all heads.  
  - The softmax dot‑product uses the **same** K/V, so we can compute it once and broadcast.

  > **Key point** – you only computed `Qh @ Kh` *once* and reused it across all heads.

#### Grouped-Query Attention (GQA)
Reduces computational load by sharing key and value matrices among groups of attention heads.
  In GQA, every *group* shares a single set of keys/values, but each group still has its own queries.  


  | Step | What happens | Why it matters |
  |------|--------------|----------------|
  | 1️⃣  **Split into heads** | The hidden vector is divided into, say, 8 heads. | Lets each head look at the data in a different way. |
  | 2️⃣  **Separate Q, K, V per head** | Each head gets its own **query**, **key**, and **value** matrices. | Gives each head a full, independent view of the sequence. |
  | 3️⃣  **Do attention per head** | Compute \(Q_i K_i^T\) for head i, then use that to weigh \(V_i\). | Classic multi‑head attention, but expensive for long sequences. |

  With *n* heads we end up with *n* distinct key/value tensors.


  | Change | What GQA does | What you gain |
  |--------|--------------|--------------|
  | **Group heads** | Heads are clustered into *G* groups (e.g., 8 heads → 2 groups of 4 heads each). | Fewer distinct key/value sets. |
  | **Shared K/V per group** | All heads in the same group use the *same* key/value matrices. | Key/value memory and computation shrink by a factor ≈ G. |
  | **Separate Q per head** | Each head still owns its own query matrix. | Each head can still learn a different “query pattern.” |
  | **Attention computation** | You compute the dot‑product once per group and broadcast it to all heads in that group. | Saves the \(L \times L\) operations that would otherwise be repeated for every head. |

  ![image-4.png](./951c27af_image-4.png)

  - **K_shared_1** and **V_shared_1** are computed once and reused by heads 1‑2.  
  - **K_shared_2** and **V_shared_2** are reused by heads 3‑4.

  > **Key point** – `K` and `V` are computed once per group and reused for every head in that group.

  Grouped-Query Attention (GQA) mechanism is often considered the best compromise, offering quality near Multi-Head Attention (MHA) while providing much of the speed and memory efficiency of MQA  

#### Flash Attention

Provides significant speedups for both training and inference of Transformer LLMs
on GPUs. It speeds up the attention calculation by optimizing what values
are loaded and moved between a GPU’s shared memory (SRAM) and high
bandwidth memory (HBM)

  - Traditional attention implementations often require storing large intermediate matrices in GPU memory, leading to high memory bandwidth usage and latency.
  - Flash Attention reorganizes the computation to minimize memory access by computing attention in smaller chunks that fit into the faster shared memory of the GPU.
  - This reduces the need to read and write large matrices to slower global memory, resulting in faster computation times and lower memory usage.
  > FlashAttention keeps the **full attention matrix** implicit, gives you the same accuracy as a standard implementation, but uses 8–10× less memory and runs 2–4× faster on a modern GPU. It's also synergic with memory-efficient attention methods like MQA and GQA, and many SOTA models use a combination of these techniques.

  [📘Flash attention paper](https://arxiv.org/abs/2205.14135)  
  [📘Flash attention 2 paper](https://tridao.me/publications/flash2/flash2.pdf)

#### Mixture of Experts (MoE)

Introduces specialized sub-networks (experts) that are activated based on the input, allowing the model to scale efficiently.

  [🐍 Mixture of Experts](../04/moe.ipynb)

#### Transformer block normalization and activation improvements

  - **Pre-Layer Normalization (Pre-LN)**:
  Pre-LN places layer normalization before the attention and feed-forward layers, leading to more stable gradients during training.

![image-5.png](./951c27af_image-5.png)

  - Another improvement is **RMSNorm**, which normalizes based on the root mean square of the inputs (instead of the mean and variance), simplifying computations and improving efficiency.
  - Lastly intead of the original **GeLU** activation function, many models now use the faster **SiLU** (or Swish) activation, which has been shown to perform better in some scenarios, or **SwiGLU**, a gated variant that enhances model capacity.

#### Positional Embeddings

Asking “Where is the word in the sentence?”, techniques like Rotary Positional Embeddings (RoPE) enhance the model's ability to understand token positions.

A transformer sees only a list of vectors (the token embeddings).  
If you swap two words, the list changes but the vectors stay the same, so the model **doesn’t know** that the order has changed.

Positional embeddings give the model a “sense of position” so that:
* “dog” at the start can be distinguished from “dog” at the end.
* The model can learn that “B A” is different from “A B”.

- The classic way – Absolute Positional Encodings:

  * Think of adding a GPS coordinate to each token.  
  * The coordinate is added once (before the first attention layer) and never changes.  
  * The model can then learn that “word i” is usually followed by “word i+1”.

  Pros: Simple, works well for short sequences.  
  Cons: Fixed length, no built‑in notion of “relative distance”.

- The modern trend – Relative / Rotary Positional Embeddings:

  * Instead of giving each token a fixed position, we teach the model to understand **how far apart** tokens are.
  * This gives a more flexible and generalisable sense of position.

  - a.  **Relative** Positional Encodings  
    * Instead of telling the model “this word is at position 5”, we tell it “this word is 2 steps ahead of the other word”.  
    * The model learns to look at **how far apart** tokens are, not just their absolute index.

  - b.  Rotary Position‑Wise Encoding (**RoPE**)

    RoPE is a clever way to inject relative distance *directly* into the dot‑product that produces attention scores.  
    It works by rotating the query and key vectors in a multi‑dimensional space:

    1. **Rotate** each query and key vector by a phase that depends on its position.  
    2. **Dot‑product** the rotated vectors – the rotation automatically encodes how far apart the positions are. 

    Because rotation is just a re‑orientation of the vectors, the rest of the transformer stays untouched.  

### Vision Transformers & multi-modal models
Vision Transformers (ViTs) adapt the transformer architecture for image data by dividing images into patches and treating them as tokens, enabling the application of transformer models to computer vision tasks.

---

#### From original paper:

> Making generation less sequential is another research goals of ours.